# Step 04b — Process Patrologia Graeca Corpus

Converts the pre-tagged `.vert` files from the Calfa/GREgORI *Patrologia Graeca* corpus
(arXiv 2603.09470, Zenodo 19915273, CC BY 4.0) into the `tagged/*.txt` format consumed
by Steps 05–06, and appends a row per volume to `tlg-texts.csv`.

**`.vert` token format** (5 tab-separated fields inside each `<w>` element):
```
OCR-form  lc-diacritic-free  lemma  lc-lemma  POS
```
Example: `τῶν  των  ὁ  ο  DET`

This step does **not** require `bert-env`.

In [1]:
import os
import re
import zipfile
import unicodedata
from pathlib import Path
from collections import Counter

import pandas as pd

PG_ZIP   = "/tmp/PG_full.zip"   # downloaded from zenodo.org/records/19915273
EXTRACT  = "/tmp/PG_full/"
TAGGED   = "./tagged/"
os.makedirs(TAGGED, exist_ok=True)

In [2]:
# Download if not present
if not os.path.exists(PG_ZIP):
    import urllib.request
    url = "https://zenodo.org/api/records/19915273/files/PG.zip/content"
    print("Downloading PG corpus from Zenodo (~103 MB)…")
    urllib.request.urlretrieve(url, PG_ZIP)
    print("Done.")
else:
    print(f"Using existing {PG_ZIP}")

# Extract
if not os.path.exists(EXTRACT):
    print("Extracting…")
    with zipfile.ZipFile(PG_ZIP) as z:
        z.extractall(EXTRACT)
    print("Done.")

vert_files = sorted(Path(EXTRACT).rglob("*.vert"))
print(f"{len(vert_files)} .vert files found")

Using existing /tmp/PG_full.zip
28 .vert files found


In [3]:
# ── Volume-level metadata ────────────────────────────────────────────────────
# Author names are simplified for use as the 'author' label in the classifier.
# Multiple authors in one volume are listed; they will appear as a single entry.
PG_METADATA = {
    "PG003":   ("Dionysius Areopagita",          "PG 3 — Dionysius Areopagita"),
    "PG005":   ("Ignatius et al.",               "PG 5 — Ignatius; Polycarp; others"),
    "PG006":   ("Justin et al.",                 "PG 6 — Justin; Tatian; Athenagoras; Theophilus; Hermias"),
    "PG009":   ("Clement of Alexandria",         "PG 9 — Clement of Alexandria (vol. 2)"),
    "PG016_3": ("Origen; Hippolytus",            "PG 16.3 — Origen (vol. 6.3); Hippolytus"),
    "PG071":   ("Cyril of Alexandria",           "PG 71 — Cyril of Alexandria (vol. 4)"),
    "PG073":   ("Cyril of Alexandria",           "PG 73 — Cyril of Alexandria (vol. 6)"),
    "PG087_1": ("Procopius of Gaza",             "PG 87.1 — Procopius of Gaza"),
    "PG101":   ("Photius",                       "PG 101 — Photius (vol. 1)"),
    "PG109":   ("Byzantine historians (10th c.)","PG 109 — Byzantine historians"),
    "PG112":   ("Constantine Porphyrogenitus",   "PG 112 — Constantine Porphyrogenitus (vol. 1)"),
    "PG113":   ("Constantine Porphyrogenitus",   "PG 113 — Constantine Porphyrogenitus (vol. 2)"),
    "PG118":   ("Oecumenius",                    "PG 118 — Oecumenius (vol. 1)"),
    "PG121":   ("George Cedrenus",               "PG 121 — George Cedrenus (vol. 1)"),
    "PG122":   ("George Cedrenus; Psellus",      "PG 122 — Cedrenus; Scylitzes; Psellus"),
    "PG123":   ("Theophylact of Bulgaria",       "PG 123 — Theophylact of Bulgaria (vol. 1)"),
    "PG124":   ("Theophylact of Bulgaria",       "PG 124 — Theophylact of Bulgaria (vol. 2)"),
    "PG125":   ("Theophylact of Bulgaria",       "PG 125 — Theophylact of Bulgaria (vol. 3)"),
    "PG126":   ("Theophylact of Bulgaria",       "PG 126 — Theophylact of Bulgaria (vol. 4)"),
    "PG134":   ("John Zonaras",                  "PG 134 — John Zonaras (vol. 1)"),
    "PG139":   ("Nicetas Choniates et al.",      "PG 139 — Isidore of Thessalonica; Nicetas Choniates"),
    "PG146":   ("Nicephorus Callistus",          "PG 146 — Nicephorus Callistus (vol. 2)"),
    "PG148":   ("Nicephorus Gregoras",           "PG 148 — Nicephorus Gregoras (vol. 1)"),
    "PG151":   ("Unknown (14th c.)",             "PG 151"),
    "PG153":   ("Unknown (14th c.)",             "PG 153"),
    "PG155":   ("Unknown (15th c.)",             "PG 155"),
    "PG157":   ("Unknown (15th c.)",             "PG 157"),
    "PG158":   ("Unknown (15th c.)",             "PG 158"),
}

def vol_key(vert_path: Path) -> str:
    """Extract PG volume key from filename, e.g. 'PG071' or 'PG016_3' from 'PG016_3_tagged_text.vert'."""
    return vert_path.stem.replace("_tagged_text", "").upper()  # e.g. PG071, PG016_3, PG087_1

print("Metadata loaded for:", sorted(PG_METADATA))

Metadata loaded for: ['PG003', 'PG005', 'PG006', 'PG009', 'PG016_3', 'PG071', 'PG073', 'PG087_1', 'PG101', 'PG109', 'PG112', 'PG113', 'PG118', 'PG121', 'PG122', 'PG123', 'PG124', 'PG125', 'PG126', 'PG134', 'PG139', 'PG146', 'PG148', 'PG151', 'PG153', 'PG155', 'PG157', 'PG158']


In [4]:
def iter_tokens(vert_path: Path):
    """Yield (wordform, pos) from a .vert file.

    .vert structure:
        <w id="...">
        OCR-form TAB lc-form TAB lemma TAB lc-lemma TAB POS
        </w>
    We grab lines that are NOT XML tags and contain tabs.
    """
    with open(vert_path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line or line.startswith("<"):
                continue
            fields = line.split("\t")
            if len(fields) < 5:
                continue
            wordform = fields[0].strip()
            pos      = fields[4].strip().lower()  # lowercase → first char matches Flair convention
            if wordform:
                yield wordform, pos


def normalize_text(text: str) -> str:
    return unicodedata.normalize("NFKC", re.sub(r"\s{2,}|\n+", " ", text)).strip()


word_re = re.compile(r"\w+")

rows = []

for vf in vert_files:
    key = vol_key(vf)
    if key not in PG_METADATA:
        print(f"  skipping {vf.name} (no metadata)")
        continue

    author, title = PG_METADATA[key]

    tokens = list(iter_tokens(vf))
    if not tokens:
        print(f"  {key}: no tokens parsed — skipping")
        continue

    # ── Write tagged file ────────────────────────────────────────────────
    file_stem = vf.stem                          # e.g. PG071_tagged_text
    tagged_path = os.path.join(TAGGED, f"{file_stem}-tagged.txt")
    if not os.path.exists(tagged_path):
        with open(tagged_path, "w", encoding="utf-8") as out:
            out.write("\n".join(f"{w}\t{p}" for w, p in tokens))

    # ── Build plain-text field ───────────────────────────────────────────
    raw_text = normalize_text(" ".join(w for w, _ in tokens))
    token_count = len(word_re.findall(raw_text))

    rows.append({
        "file":         file_stem,
        "orig_author":  author,
        "author":       author,
        "title":        title,
        "textgroup":    f"pg{key[2:].lower()}",   # e.g. pg071
        "tokens":       token_count,
        "full-text-raw": raw_text,
    })
    print(f"  {key}: {token_count:,} tokens → {tagged_path}")

print(f"\n{len(rows)} PG volumes processed.")

  PG003: 160,273 tokens → ./tagged/PG003_tagged_text-tagged.txt
  PG005: 47,804 tokens → ./tagged/PG005_tagged_text-tagged.txt


  PG006: 213,842 tokens → ./tagged/PG006_tagged_text-tagged.txt
  PG009: 102,961 tokens → ./tagged/PG009_tagged_text-tagged.txt


  PG016_3: 72,758 tokens → ./tagged/PG016_3_tagged_text-tagged.txt


  PG071: 211,135 tokens → ./tagged/PG071_tagged_text-tagged.txt


  PG073: 228,222 tokens → ./tagged/PG073_tagged_text-tagged.txt


...

  PG148: 235,047 tokens → ./tagged/PG148_tagged_text-tagged.txt


  PG151: 399,681 tokens → ./tagged/PG151_tagged_text-tagged.txt


  PG153: 230,406 tokens → ./tagged/PG153_tagged_text-tagged.txt


  PG155: 199,967 tokens → ./tagged/PG155_tagged_text-tagged.txt
  PG157: 123,853 tokens → ./tagged/PG157_tagged_text-tagged.txt


  PG158: 197,492 tokens → ./tagged/PG158_tagged_text-tagged.txt

28 PG volumes processed.


In [5]:
# ── Merge into tlg-texts.csv ─────────────────────────────────────────────────
pg_df = pd.DataFrame(rows)

existing_path = "tlg-texts.csv"
if os.path.exists(existing_path):
    existing = pd.read_csv(existing_path)
    # Avoid duplicating PG rows if cell is re-run
    existing = existing[~existing["file"].isin(pg_df["file"])]
    combined = pd.concat([existing, pg_df], ignore_index=True)
else:
    combined = pg_df

combined.to_csv(existing_path, index=False)
print(f"tlg-texts.csv: {len(combined)} rows total ({len(pg_df)} PG, {len(combined)-len(pg_df)} First1KGreek)")
combined[["file","author","tokens"]].tail(len(pg_df))

tlg-texts.csv: 632 rows total (28 PG, 604 First1KGreek)


,file,author,tokens
604,PG003_tagged_text,Dionysius Areopagita,160273
605,PG005_tagged_text,Ignatius et al.,47804
606,PG006_tagged_text,Justin et al.,213842
607,PG009_tagged_text,Clement of Alexandria,102961
608,PG016_3_tagged_text,Origen; Hippolytus,72758
609,PG071_tagged_text,Cyril of Alexandria,211135
610,PG073_tagged_text,Cyril of Alexandria,228222
611,PG087_1_tagged_text,Procopius of Gaza,208057
612,PG101_tagged_text,Photius,232486
613,PG109_tagged_text,Byzantine historians (10th c.),209559
